In [57]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import json , ast
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
import torch
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight
from transformers import DataCollatorForSeq2Seq
import torch.nn as nn  
from sklearn.metrics import f1_score,accuracy_score,classification_report

In [2]:
pip install 'accelerate>=1.1.0'

Note: you may need to restart the kernel to use updated packages.


In [3]:
SEED = 42

In [4]:
train = pd.read_excel(r'DeepX_train.xlsx')

unlabeled = pd.read_excel(r'DeepX_unlabeled.xlsx')

validation = pd.read_excel(r'DeepX_validation.xlsx')

In [5]:
train.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [6]:
validation.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,4446,مريم سوتلي الاظافررر تحفههه اوييي ❤️❤️❤️❤️❤️,5,قبل شهرين,Sand salon,صالون تجميل,google_maps,"[""service""]","{""service"": ""positive""}"
1,8612,التطبيق جميل .. أتمنى إضافة البحث عن طريق الخر...,4,2020-10-28 00:00:00,Aqarmap,real_estate,play_store,"[""app_experience""]","{""app_experience"": ""neutral""}"
2,6729,سراقين مكتوب وصلت السياره والسواق مارضى يقول و...,1,2026-02-04 00:00:00,Careem,transport,play_store,"[""service"", ""delivery"", ""price""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,6292,سي جيدا,1,2025-08-07 00:00:00,Elmenus,food_delivery,play_store,"[""general""]","{""general"": ""negative""}"
4,1639,مكان ممتاز جدا و الخدمة جيده جدا,4,قبل أسبوع,Holm Cafe,مقهى,google_maps,"[""ambiance"", ""service""]","{""ambiance"": ""positive"", ""service"": ""positive""}"


In [7]:
unlabeled.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform
0,1,Incroyablement grand avec des belles boutiques...,5,قبل 7 ساعات,مول سيتي ستارز.,مركز تسوق,google_maps
1,2,زحمه جدا,5,قبل 12 ساعة,مول سيتي ستارز.,مركز تسوق,google_maps
2,3,حلو فخم كشخة محترم ورايق ينفع للعوائل الخليجي...,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق,google_maps
3,4,طبعا غني عن التعريف بتاع البشوات,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق,google_maps
4,5,Centro commerciale al Cairo... Molto grande e ...,5,قبل يومين (2),مول سيتي ستارز.,مركز تسوق,google_maps


# Constants 

In [8]:
ASPECTS = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general"
]

SENTIMENTS = ["negative", "neutral", "positive"]

sentiment2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2sentiment = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

aspect_ar = {
    "food": "الطعام",
    "service": "الخدمة",
    "price": "السعر",
    "cleanliness": "النظافة",
    "delivery": "التوصيل",
    "ambiance": "المكان والجو العام",
    "app_experience": "تجربة التطبيق",
    "general": "التقييم العام"
}

# Clean the dataset

In [9]:
def clean_text(text):
    text = str(text)

    text = text.replace("ـ", "")
    text = re.sub(r"[\u064B-\u065F]", "", text)

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    text = re.sub(r"http\S+|www\S+", " رابط ", text)
    text = re.sub(r"@\w+", " مستخدم ", text)

    # Keeps Arabic, English, numbers
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# Parsing and Nulls Handling 

In [10]:
def parse_obj(x):
    if isinstance(x, (list, dict)):
        return x

    if pd.isna(x):
        return None

    x = str(x).strip()

    try:
        return json.loads(x)
    except:
        pass

    try:
        return ast.literal_eval(x)
    except:
        pass

    if "," in x:
        return [item.strip() for item in x.split(",")]

    return x

# Remove the Exact duplicates in the data 

In [11]:
train_raw = train.drop_duplicates(
    subset=["review_text", "aspects", "aspect_sentiments"]
).reset_index(drop=True)


# Explode Labeled data into aspects level raws

In [12]:
def explode_train(df):
    rows = []

    for _, row in df.iterrows():
        review_id = row["review_id"]
        review_text = str(row["review_text"])

        aspects = parse_obj(row["aspects"])
        aspect_sentiments = parse_obj(row["aspect_sentiments"])

        if aspects is None:
            continue

        if isinstance(aspects, str):
            aspects = [aspects]

        if not isinstance(aspect_sentiments, dict):
            continue

        for aspect in aspects:
            aspect = str(aspect).strip()

            if aspect == "none":
                continue

            if aspect not in ASPECTS:
                continue

            sentiment = aspect_sentiments.get(aspect)

            if sentiment not in SENTIMENTS:
                continue

            rows.append({
                "review_id": review_id,
                "review_text": review_text,
                "star_rating": row.get("star_rating", ""),
                "platform": row.get("platform", ""),
                "business_category": row.get("business_category", ""),
                "aspect": aspect,
                "sentiment": sentiment
            })

    return pd.DataFrame(rows)


df = explode_train(train_raw)

df["review_text"] = df["review_text"].apply(clean_text)
df["label"] = df["sentiment"].map(sentiment2id)

print("Aspect-level shape:", df.shape)
print(df.head())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nAspect-sentiment distribution:")
print(df.groupby(["aspect", "sentiment"]).size().sort_values())

Aspect-level shape: (3191, 8)
   review_id                                        review_text  star_rating  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
1       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
2       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
3       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
4       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   

      platform business_category          aspect sentiment  label  
0   play_store         ecommerce  app_experience  negative      0  
1   play_store         ecommerce        delivery  negative      0  
2  google_maps             كافيه     cleanliness  positive      2  
3  google_maps             كافيه        ambiance  positive      2  
4  google_maps             كافيه         service  positive      2  

Sentiment distribution:
sentiment
positive    1565
negative    1536
neutral     

# Build the model input shape 

In [13]:
def build_input(row):
    aspect_name = aspect_ar.get(row["aspect"], row["aspect"])

    return (
        f"التقييم: {row.get('star_rating', '')} [SEP] "
        f"المنصة: {row.get('platform', '')} [SEP] "
        f"النشاط: {row.get('business_category', '')} [SEP] "
        f"الجانب: {aspect_name} [SEP] "
        f"النص: {row['review_text']}"
    )


df["input_text"] = df.apply(build_input, axis=1)

df[["input_text", "sentiment"]].head()

,input_text,sentiment
0,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...,negative
1,التقييم: 3 [SEP] المنصة: play_store [SEP] النش...,negative
2,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive
3,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive
4,التقييم: 5 [SEP] المنصة: google_maps [SEP] الن...,positive


# Split by rev id not by exploded 

why ? 
If you split exploded rows randomly, the same review may appear in both train and validation.

In [14]:
unique_ids = df["review_id"].unique()

train_ids, valid_ids = train_test_split(
    unique_ids,
    test_size=0.15,
    random_state=SEED
)

train_df = df[df["review_id"].isin(train_ids)].reset_index(drop=True)
valid_df = df[df["review_id"].isin(valid_ids)].reset_index(drop=True)

print("Train aspect rows:", train_df.shape)
print("Valid aspect rows:", valid_df.shape)

print("\nTrain sentiment distribution:")
print(train_df["sentiment"].value_counts())

print("\nValid sentiment distribution:")
print(valid_df["sentiment"].value_counts())

Train aspect rows: (2736, 9)
Valid aspect rows: (455, 9)

Train sentiment distribution:
sentiment
negative    1333
positive    1324
neutral       79
Name: count, dtype: int64

Valid sentiment distribution:
sentiment
positive    241
negative    203
neutral      11
Name: count, dtype: int64


# Prepare text for Masked LLM

In [15]:
mlm_texts = pd.concat([
    train_raw["review_text"],
    unlabeled["review_text"]
], ignore_index=True)

mlm_texts = (
    mlm_texts
    .dropna()
    .astype(str)
    .apply(clean_text)
)

mlm_texts = mlm_texts[mlm_texts.str.len() > 2]
mlm_texts = mlm_texts.drop_duplicates().reset_index(drop=True)

print("MLM texts:", len(mlm_texts))
mlm_texts.head()

MLM texts: 7672


0                  لا يوجد الدفع بالبطاقه عند الاستلام
1    المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...
2    تجربة سيية سالتهم الاكل هياخد وقت قد ايه قالول...
3                                      احلي مكان فزايد
4    الفطير حلو جدا الاحجام تحفة بالنسبه للسعر فا ي...
Name: review_text, dtype: object

# Define models we will be using 

In [16]:
BASE_MLM_MODEL = "xlm-roberta-base"
DOMAIN_MODEL_DIR = "./domain_xlm_roberta_reviews"

## Tokenize data for model 

In [17]:
mlm_tokenizer = AutoTokenizer.from_pretrained(BASE_MLM_MODEL)

mlm_dataset = Dataset.from_dict({
    "text": mlm_texts.tolist()
})

def tokenize_for_mlm(batch):
    return mlm_tokenizer(
        batch["text"],
        truncation=True,
        max_length=160
    )

mlm_dataset = mlm_dataset.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/7672 [00:00<?, ? examples/s]

## init the data collator 

In [18]:
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=mlm_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Train the model 

In [19]:
# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected. Training will use CPU (much slower).")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
CUDA version: 12.8


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [21]:
device = torch.device("cuda")

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [23]:
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MLM_MODEL)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlm_model = mlm_model.to(device)
print(f"Model moved to: {device}")

mlm_args = TrainingArguments(
    output_dir=DOMAIN_MODEL_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    gradient_checkpointing=True,
    use_cpu=False
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_dataset,
    data_collator=mlm_collator
)

mlm_trainer.train()

mlm_trainer.save_model(DOMAIN_MODEL_DIR)
mlm_tokenizer.save_pretrained(DOMAIN_MODEL_DIR)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model moved to: cuda


Step,Training Loss
100,18.144934
200,16.448231
300,14.971780
400,15.902971
500,14.695743
600,14.673326
700,14.091210
800,15.245784
900,14.673623
1000,13.995403


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./domain_xlm_roberta_reviews/tokenizer_config.json',
 './domain_xlm_roberta_reviews/tokenizer.json')

# Use weighted class approach for non balanced data 


In [24]:
# Use class weights because of the non balance bet data 
def get_class_weights(labels):
    classes = np.array([0, 1, 2])

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=labels
    )

    return torch.tensor(weights, dtype=torch.float)

In [25]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = self.class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 3), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [26]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds)
    }

# Create the sentiment transformer 

In [49]:
def train_transformer_model(
    model_name,
    train_df,
    valid_df,
    output_dir,
    epochs=4,
    lr=2e-5,
    batch_size=8,
    max_length=160
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_ds = Dataset.from_pandas(
        train_df[["input_text", "label"]].reset_index(drop=True)
    )

    valid_ds = Dataset.from_pandas(
        valid_df[["input_text", "label"]].reset_index(drop=True)
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=max_length
        )

    train_ds = train_ds.map(tokenize_batch, batched=True)
    valid_ds = valid_ds.map(tokenize_batch, batched=True)

    train_ds = train_ds.rename_column("label", "labels")
    valid_ds = valid_ds.rename_column("label", "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2sentiment,
        label2id=sentiment2id
    )

    class_weights = get_class_weights(train_df["label"].values)

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    return trainer, tokenizer

# Prop prediction function 

In [37]:
def predict_proba_transformer(trainer, tokenizer, texts, max_length=160):
    ds = Dataset.from_dict({
        "input_text": list(texts)
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=max_length
        )

    ds = ds.map(tokenize_batch, batched=True)

    preds = trainer.predict(ds)
    logits = preds.predictions

    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

    return probs

# Train domain model xlim roberta 

In [56]:
trainer_xlm, tokenizer_xlm = train_transformer_model(
    model_name=DOMAIN_MODEL_DIR,
    train_df=train_df,
    valid_df=valid_df,
    output_dir="./sentiment_domain_xlm",
    epochs=4,
    lr=2e-5,
    batch_size=8,
    max_length=160
)

valid_probs_xlm = predict_proba_transformer(
    trainer_xlm,
    tokenizer_xlm,
    valid_df["input_text"]
)

valid_pred_xlm = valid_probs_xlm.argmax(axis=1)

print("Domain XLM-R Macro F1:", f1_score(valid_df["label"], valid_pred_xlm, average="macro"))
print(classification_report(valid_df["label"], valid_pred_xlm, target_names=SENTIMENTS))

Map:   0%|          | 0/2736 [00:00<?, ? examples/s]

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: ./domain_xlm_roberta_reviews
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.812290,1.178619,0.600839,0.892308
2,0.676037,0.955392,0.612168,0.907692
3,0.555143,1.102299,0.615046,0.912088
4,1.022118,0.972728,0.621051,0.920879


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

Domain XLM-R Macro F1: 0.6210513052618315


NameError: name 'classification_report' is not defined

In [58]:
valid_probs_xlm = predict_proba_transformer(
    trainer_xlm,
    tokenizer_xlm,
    valid_df["input_text"]
)

valid_pred_xlm = valid_probs_xlm.argmax(axis=1)

print("Domain XLM-R Macro F1:", f1_score(valid_df["label"], valid_pred_xlm, average="macro"))
print(classification_report(valid_df["label"], valid_pred_xlm, target_names=SENTIMENTS))

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

Domain XLM-R Macro F1: 0.6210513052618315
              precision    recall  f1-score   support

    negative       0.90      0.95      0.92       203
     neutral       0.00      0.00      0.00        11
    positive       0.94      0.94      0.94       241

    accuracy                           0.92       455
   macro avg       0.61      0.63      0.62       455
weighted avg       0.90      0.92      0.91       455



/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_sta

# Third model

In [59]:
MARBERT_MODEL = "UBC-NLP/MARBERTv2"

In [61]:
trainer_marbert, tokenizer_marbert = train_transformer_model(
    model_name=MARBERT_MODEL,
    train_df=train_df,
    valid_df=valid_df,
    output_dir="./sentiment_marbert",
    epochs=4,
    lr=2e-5,
    batch_size=8,
    max_length=160
)

Map:   0%|          | 0/2736 [00:00<?, ? examples/s]

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.759201,0.890835,0.616643,0.914286
2,0.561705,0.774891,0.619733,0.918681
3,0.396587,0.885141,0.632875,0.938462
4,0.719513,0.802396,0.751729,0.940659


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [62]:
valid_probs_marbert = predict_proba_transformer(
    trainer_marbert,
    tokenizer_marbert,
    valid_df["input_text"]
)

valid_pred_marbert = valid_probs_marbert.argmax(axis=1)

print("MARBERT Macro F1:", f1_score(valid_df["label"], valid_pred_marbert, average="macro"))
print(classification_report(valid_df["label"], valid_pred_marbert, target_names=SENTIMENTS))

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

MARBERT Macro F1: 0.7517292365950276
              precision    recall  f1-score   support

    negative       0.93      0.96      0.94       203
     neutral       0.50      0.27      0.35        11
    positive       0.96      0.96      0.96       241

    accuracy                           0.94       455
   macro avg       0.80      0.73      0.75       455
weighted avg       0.94      0.94      0.94       455



# Fourth model 

In [63]:
CAMEL_MODEL = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

In [64]:
trainer_camel, tokenizer_camel = train_transformer_model(
    model_name=CAMEL_MODEL,
    train_df=train_df,
    valid_df=valid_df,
    output_dir="./sentiment_camelbert",
    epochs=4,
    lr=2e-5,
    batch_size=8,
    max_length=160
)

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2736 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.783846,0.762333,0.618000,0.916484
2,0.631877,0.803535,0.613712,0.909890
3,0.442997,0.895091,0.623872,0.925275
4,0.793691,0.876825,0.623606,0.916484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [65]:
valid_probs_camel = predict_proba_transformer(
    trainer_camel,
    tokenizer_camel,
    valid_df["input_text"]
)

valid_pred_camel = valid_probs_camel.argmax(axis=1)

print("CAMeLBERT Macro F1:", f1_score(valid_df["label"], valid_pred_camel, average="macro"))
print(classification_report(valid_df["label"], valid_pred_camel, target_names=SENTIMENTS))

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

CAMeLBERT Macro F1: 0.6238715147887675
              precision    recall  f1-score   support

    negative       0.92      0.94      0.93       203
     neutral       0.00      0.00      0.00        11
    positive       0.93      0.96      0.94       241

    accuracy                           0.93       455
   macro avg       0.62      0.63      0.62       455
weighted avg       0.90      0.93      0.91       455



/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_sta

In [66]:
f1_xlm = f1_score(valid_df["label"], valid_pred_xlm, average="macro")
f1_marbert = f1_score(valid_df["label"], valid_pred_marbert, average="macro")
f1_camel = f1_score(valid_df["label"], valid_pred_camel, average="macro")

scores = np.array([f1_xlm, f1_marbert, f1_camel])
weights = scores / scores.sum()

print("Scores:", scores)
print("Weights:", weights)

valid_probs_ensemble = (
    weights[0] * valid_probs_xlm +
    weights[1] * valid_probs_marbert +
    weights[2] * valid_probs_camel
)

valid_pred_ensemble = valid_probs_ensemble.argmax(axis=1)

print("Weighted Ensemble Macro F1:", f1_score(valid_df["label"], valid_pred_ensemble, average="macro"))
print(classification_report(valid_df["label"], valid_pred_ensemble, target_names=SENTIMENTS))

Scores: [0.62105131 0.75172924 0.62387151]
Weights: [0.31104634 0.37649486 0.3124588 ]
Weighted Ensemble Macro F1: 0.6261428761828602
              precision    recall  f1-score   support

    negative       0.92      0.94      0.93       203
     neutral       0.00      0.00      0.00        11
    positive       0.94      0.96      0.95       241

    accuracy                           0.93       455
   macro avg       0.62      0.63      0.63       455
weighted avg       0.91      0.93      0.92       455

